# XGBoost - Notebook Autonome (Rendu)

Ce notebook est **autonome**: le code XGBoost est embarqué dans les cellules
et exécuté directement dans le notebook (sans appel `!python train_xgboost.py`).

Note importante: ce workflow a été principalement développé et validé en local Windows/PowerShell.


## Cellule 1 - Installation (si besoin)

Installe les dépendances minimales. Ignore cette cellule si ton environnement est déjà prêt.


In [ ]:
# %pip install -q numpy pandas matplotlib scikit-learn xgboost imbalanced-learn


## Cellule 2 - Chemins projet/données

Adapte ces chemins selon ton environnement local ou Colab/Drive.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
INDICES_CSV = PROJECT_ROOT / "data" / "s2_herault_2024_full" / "indices_parcelles_2024-01-01_2024-12-31_win5d.csv"
LABELS_CSV = PROJECT_ROOT / "export.csv"
OUTPUT_ROOT = PROJECT_ROOT / "outputs_xgboost_notebook"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INDICES_CSV:", INDICES_CSV)
print("LABELS_CSV:", LABELS_CSV)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Cellule 3 - Chargement du pipeline XGBoost embarqué

Le script `train_xgboost.py` est embarqué dans un module mémoire et exécuté depuis le notebook.


In [ ]:
import sys
import types

XGBOOST_SOURCE = '#!/usr/bin/env python\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import clone\nfrom sklearn.metrics import (\n    accuracy_score,\n    balanced_accuracy_score,\n    classification_report,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n)\nfrom sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split\nfrom sklearn.preprocessing import LabelEncoder\n\ntry:\n    from xgboost import XGBClassifier\nexcept ImportError as exc:\n    raise SystemExit(\n        "xgboost n\'est pas installe. Lance: pip install xgboost scikit-learn"\n    ) from exc\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Classification des cultures avec XGBoost a partir d\'indices Sentinel-2"\n    )\n    parser.add_argument(\n        "--indices-csv",\n        default="data/s2_herault_2024_full/indices_parcelles_2024-01-01_2024-12-31_win5d.csv",\n        help="CSV long des indices par parcelle",\n    )\n    parser.add_argument("--labels-csv", default="export.csv", help="CSV RPG (verite terrain)")\n    parser.add_argument("--id-col", default="ID_PARCEL", help="Nom de la colonne identifiant parcelle")\n    parser.add_argument("--target-col", default="CODE_CULTU", help="Nom de la colonne cible")\n    parser.add_argument(\n        "--indices", nargs="+", default=["NDVI", "EVI", "NDMI", "NDWI"], help="Indices a conserver"\n    )\n    parser.add_argument("--cloud-max", type=float, default=40.0, help="Seuil max cloud_scene")\n    parser.add_argument("--px-count-min", type=int, default=5, help="Seuil min de pixels valides")\n    parser.add_argument(\n        "--min-class-count",\n        type=int,\n        default=200,\n        help="Conserver les classes avec au moins N parcelles",\n    )\n    parser.add_argument("--test-size", type=float, default=0.2, help="Proportion du test set")\n    parser.add_argument(\n        "--val-size",\n        type=float,\n        default=0.15,\n        help="Proportion de validation interne prelevee sur le train",\n    )\n    parser.add_argument("--random-state", type=int, default=42, help="Graine aleatoire")\n    parser.add_argument(\n        "--augment",\n        choices=["none", "smote", "borderline_smote"],\n        default="none",\n        help="Augmentation du train set (apres split): none, smote, borderline_smote",\n    )\n    parser.add_argument(\n        "--smote-k-neighbors",\n        type=int,\n        default=5,\n        help="Nombre de voisins pour SMOTE/BorderlineSMOTE",\n    )\n    parser.add_argument(\n        "--cv-folds",\n        type=int,\n        default=5,\n        help="Nombre de folds pour validation croisee stratifiee (>=2, 0 pour desactiver)",\n    )\n    parser.add_argument(\n        "--tune-n-iter",\n        type=int,\n        default=15,\n        help="Nombre d\'iterations RandomizedSearchCV (0 pour desactiver)",\n    )\n    parser.add_argument(\n        "--tune-verbose",\n        type=int,\n        default=1,\n        help="Verbosite de RandomizedSearchCV",\n    )\n    parser.add_argument(\n        "--tune-n-jobs",\n        type=int,\n        default=1,\n        help="Nombre de jobs paralleles pour RandomizedSearchCV (1 recommande sous Windows)",\n    )\n    parser.add_argument(\n        "--train-n-jobs",\n        type=int,\n        default=-1,\n        help="Nombre de threads XGBoost pour l\'entrainement final (-1 = tous les coeurs)",\n    )\n    parser.add_argument(\n        "--early-stopping-rounds",\n        type=int,\n        default=50,\n        help="Nombre de rounds pour early stopping (0 pour desactiver)",\n    )\n    parser.add_argument("--max-depth", type=int, default=6, help="Profondeur max des arbres")\n    parser.add_argument("--min-child-weight", type=float, default=4.0, help="Poids mini par noeud feuille")\n    parser.add_argument("--gamma", type=float, default=0.1, help="Gain mini pour split")\n    parser.add_argument("--reg-alpha", type=float, default=0.1, help="Regularisation L1")\n    parser.add_argument("--reg-lambda", type=float, default=2.0, help="Regularisation L2")\n    parser.add_argument(\n        "--disable-phenology",\n        action="store_true",\n        help="Desactive les features phenologiques (pic, amplitude, AUC, etc.)",\n    )\n    parser.add_argument(\n        "--max-dates",\n        type=int,\n        default=0,\n        help="Nombre max de dates a conserver (0 = toutes)",\n    )\n    parser.add_argument("--output-dir", default="outputs_xgboost", help="Dossier de sortie")\n    return parser.parse_args()\n\n\ndef load_labels(path: Path, id_col: str, target_col: str) -> pd.DataFrame:\n    labels = pd.read_csv(path, usecols=[id_col, target_col])\n    labels = labels.dropna(subset=[target_col]).copy()\n    labels[id_col] = labels[id_col].astype(str)\n    labels[target_col] = labels[target_col].astype(str)\n    labels = labels.drop_duplicates(subset=[id_col])\n    return labels\n\n\ndef aggregate_indices(\n    path: Path,\n    id_set: set,\n    indices_keep: set,\n    cloud_max: float,\n    px_count_min: int,\n    chunksize: int = 500_000,\n) -> pd.DataFrame:\n    cols = ["date", "ID_PARCEL", "index", "value_mean", "px_count", "cloud_scene"]\n    agg_chunks = []\n\n    for chunk in pd.read_csv(path, usecols=cols, chunksize=chunksize):\n        chunk = chunk.dropna(subset=["date", "ID_PARCEL", "index", "value_mean", "px_count", "cloud_scene"])\n        chunk["ID_PARCEL"] = chunk["ID_PARCEL"].astype(str)\n        chunk = chunk[chunk["ID_PARCEL"].isin(id_set)]\n        chunk = chunk[chunk["index"].isin(indices_keep)]\n        chunk = chunk[chunk["cloud_scene"] <= cloud_max]\n        chunk = chunk[chunk["px_count"] >= px_count_min]\n        if chunk.empty:\n            continue\n\n        chunk["weighted"] = chunk["value_mean"] * chunk["px_count"]\n        g = (\n            chunk.groupby(["ID_PARCEL", "date", "index"], as_index=False)\n            .agg(weighted_sum=("weighted", "sum"), px_sum=("px_count", "sum"))\n        )\n        agg_chunks.append(g)\n\n    if not agg_chunks:\n        raise RuntimeError("Aucune ligne conservee apres filtrage des indices.")\n\n    agg = pd.concat(agg_chunks, ignore_index=True)\n    agg = (\n        agg.groupby(["ID_PARCEL", "date", "index"], as_index=False)\n        .agg(weighted_sum=("weighted_sum", "sum"), px_sum=("px_sum", "sum"))\n    )\n    agg["value"] = agg["weighted_sum"] / agg["px_sum"]\n    return agg[["ID_PARCEL", "date", "index", "value"]]\n\n\ndef limit_dates(agg: pd.DataFrame, max_dates: int) -> pd.DataFrame:\n    if max_dates <= 0:\n        return agg\n\n    date_counts = agg.groupby("date")["ID_PARCEL"].nunique().sort_values(ascending=False)\n    keep_dates = set(date_counts.head(max_dates).index)\n    return agg[agg["date"].isin(keep_dates)].copy()\n\n\ndef build_wide_features(agg: pd.DataFrame) -> pd.DataFrame:\n    feat_name = agg["index"].astype(str) + "__" + agg["date"].astype(str)\n    tmp = agg[["ID_PARCEL", "value"]].copy()\n    tmp["feature"] = feat_name\n\n    wide = tmp.pivot_table(index="ID_PARCEL", columns="feature", values="value", aggfunc="mean")\n    wide.columns.name = None\n    wide = wide.reset_index()\n    return wide\n\n\ndef build_phenology_features(agg: pd.DataFrame) -> pd.DataFrame:\n    gkeys = ["ID_PARCEL", "index"]\n    base = agg[["ID_PARCEL", "index", "date", "value"]].copy()\n    base["date_dt"] = pd.to_datetime(base["date"], errors="coerce")\n    base = base.dropna(subset=["date_dt"]).copy()\n    base["doy"] = base["date_dt"].dt.dayofyear.astype(float)\n\n    stats = (\n        base.groupby(gkeys)["value"]\n        .agg(["mean", "std", "min", "max", "median"])\n        .reset_index()\n    )\n    stats["amp"] = stats["max"] - stats["min"]\n\n    idx_max = base.groupby(gkeys)["value"].idxmax()\n    idx_min = base.groupby(gkeys)["value"].idxmin()\n    peak = base.loc[idx_max, gkeys + ["doy"]].rename(columns={"doy": "doy_peak"})\n    trough = base.loc[idx_min, gkeys + ["doy"]].rename(columns={"doy": "doy_trough"})\n\n    srt = base.sort_values(gkeys + ["doy"]).copy()\n    srt["doy_next"] = srt.groupby(gkeys)["doy"].shift(-1)\n    srt["val_next"] = srt.groupby(gkeys)["value"].shift(-1)\n    srt["auc_seg"] = 0.5 * (srt["value"] + srt["val_next"]) * (srt["doy_next"] - srt["doy"])\n    auc = srt.groupby(gkeys, as_index=False)["auc_seg"].sum().rename(columns={"auc_seg": "auc"})\n\n    pheno = stats.merge(peak, on=gkeys, how="left").merge(trough, on=gkeys, how="left").merge(auc, on=gkeys, how="left")\n    pheno_wide = pheno.set_index(gkeys).unstack("index")\n    pheno_wide.columns = [f"{idx}__{feat}" for feat, idx in pheno_wide.columns]\n    pheno_wide = pheno_wide.reset_index()\n    return pheno_wide\n\n\ndef interpolate_by_index(X: pd.DataFrame) -> pd.DataFrame:\n    cols = [c for c in X.columns if "__" in c]\n    temporal_cols = []\n    for col in cols:\n        _, suffix = col.split("__", 1)\n        parsed = pd.to_datetime(suffix, errors="coerce")\n        if not pd.isna(parsed):\n            temporal_cols.append(col)\n\n    grouped = {}\n    for col in temporal_cols:\n        idx_name, date_str = col.split("__", 1)\n        grouped.setdefault(idx_name, []).append((date_str, col))\n\n    for idx_name, items in grouped.items():\n        sorted_cols = [c for _, c in sorted(items, key=lambda x: x[0])]\n        X[sorted_cols] = X[sorted_cols].interpolate(axis=1, limit_direction="both")\n\n    numeric_cols = X.select_dtypes(include=[np.number]).columns\n    medians = X[numeric_cols].median(axis=0)\n    X[numeric_cols] = X[numeric_cols].fillna(medians)\n    return X\n\n\ndef augment_train_data(\n    X_train: pd.DataFrame,\n    y_train: np.ndarray,\n    method: str,\n    random_state: int,\n    k_neighbors: int,\n) -> tuple[pd.DataFrame, np.ndarray]:\n    if method == "none":\n        return X_train, y_train\n\n    try:\n        if method == "smote":\n            from imblearn.over_sampling import SMOTE\n\n            sampler = SMOTE(random_state=random_state, k_neighbors=k_neighbors)\n        elif method == "borderline_smote":\n            from imblearn.over_sampling import BorderlineSMOTE\n\n            sampler = BorderlineSMOTE(random_state=random_state, k_neighbors=k_neighbors)\n        else:\n            raise ValueError(f"Methode d\'augmentation inconnue: {method}")\n    except ImportError as exc:\n        raise SystemExit(\n            "imbalanced-learn n\'est pas installe. Lance: pip install imbalanced-learn"\n        ) from exc\n\n    X_res, y_res = sampler.fit_resample(X_train, y_train)\n    if not isinstance(X_res, pd.DataFrame):\n        X_res = pd.DataFrame(X_res, columns=X_train.columns)\n    return X_res, y_res\n\n\ndef make_xgb_model(\n    n_classes: int,\n    random_state: int,\n    n_jobs: int,\n    max_depth: int,\n    min_child_weight: float,\n    gamma: float,\n    reg_alpha: float,\n    reg_lambda: float,\n    overrides: dict | None = None,\n) -> XGBClassifier:\n    params = {\n        "objective": "multi:softprob",\n        "num_class": n_classes,\n        "n_estimators": 600,\n        "learning_rate": 0.05,\n        "max_depth": max_depth,\n        "min_child_weight": min_child_weight,\n        "subsample": 0.8,\n        "colsample_bytree": 0.8,\n        "gamma": gamma,\n        "reg_alpha": reg_alpha,\n        "reg_lambda": reg_lambda,\n        "tree_method": "hist",\n        "random_state": random_state,\n        "n_jobs": n_jobs,\n        "eval_metric": "mlogloss",\n    }\n    if overrides:\n        params.update(overrides)\n    return XGBClassifier(**params)\n\n\ndef stratified_cv_macro_f1(\n    model: XGBClassifier,\n    X_train: pd.DataFrame,\n    y_train: np.ndarray,\n    sample_weight: np.ndarray,\n    cv_folds: int,\n    random_state: int,\n) -> tuple[float, float]:\n    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=random_state)\n    scores = []\n    for tr_idx, va_idx in skf.split(X_train, y_train):\n        fold_model = clone(model)\n        X_tr = X_train.iloc[tr_idx]\n        X_va = X_train.iloc[va_idx]\n        y_tr = y_train[tr_idx]\n        y_va = y_train[va_idx]\n        w_tr = sample_weight[tr_idx]\n        fold_model.fit(X_tr, y_tr, sample_weight=w_tr, verbose=False)\n        pred = fold_model.predict(X_va)\n        scores.append(f1_score(y_va, pred, average="macro"))\n    return float(np.mean(scores)), float(np.std(scores))\n\n\ndef aggregate_feature_importance(model: XGBClassifier, feature_names: list[str]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    booster = model.get_booster()\n    raw_gain = booster.get_score(importance_type="gain")\n\n    gain_rows = []\n    for k, v in raw_gain.items():\n        if k.startswith("f") and k[1:].isdigit():\n            idx = int(k[1:])\n            if idx < len(feature_names):\n                name = feature_names[idx]\n            else:\n                continue\n        else:\n            name = k\n        gain_rows.append((name, float(v)))\n\n    if not gain_rows:\n        empty = pd.DataFrame(columns=["feature", "gain"])\n        return empty, pd.DataFrame(columns=["date", "gain"]), pd.DataFrame(columns=["index", "gain"])\n\n    feat_imp = pd.DataFrame(gain_rows, columns=["feature", "gain"]).sort_values("gain", ascending=False)\n\n    tmp = feat_imp.copy()\n    tmp[["index", "suffix"]] = tmp["feature"].str.split("__", n=1, expand=True)\n    tmp["date"] = pd.to_datetime(tmp["suffix"], errors="coerce")\n\n    date_imp = (\n        tmp.dropna(subset=["date"])\n        .assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d"))\n        .groupby("date", as_index=False)["gain"]\n        .sum()\n        .sort_values("gain", ascending=False)\n    )\n    index_imp = tmp.groupby("index", as_index=False)["gain"].sum().sort_values("gain", ascending=False)\n    return feat_imp, date_imp, index_imp\n\n\ndef main() -> None:\n    args = parse_args()\n    out_dir = Path(args.output_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n\n    labels = load_labels(Path(args.labels_csv), args.id_col, args.target_col)\n    id_set = set(labels[args.id_col].tolist())\n\n    print(f"Labels: {len(labels):,} parcelles, {labels[args.target_col].nunique()} classes")\n\n    agg = aggregate_indices(\n        path=Path(args.indices_csv),\n        id_set=id_set,\n        indices_keep=set(args.indices),\n        cloud_max=args.cloud_max,\n        px_count_min=args.px_count_min,\n    )\n    agg = limit_dates(agg, args.max_dates)\n    print(f"Indices agreges: {len(agg):,} lignes, {agg[\'date\'].nunique()} dates, {agg[\'index\'].nunique()} indices")\n\n    X_wide = build_wide_features(agg).rename(columns={"ID_PARCEL": args.id_col})\n    data = labels.merge(X_wide, on=args.id_col, how="inner")\n\n    if not args.disable_phenology:\n        pheno_wide = build_phenology_features(agg).rename(columns={"ID_PARCEL": args.id_col})\n        data = data.merge(pheno_wide, on=args.id_col, how="left")\n\n    class_counts = data[args.target_col].value_counts()\n    keep_classes = class_counts[class_counts >= args.min_class_count].index\n    data = data[data[args.target_col].isin(keep_classes)].copy()\n\n    if data.empty:\n        raise RuntimeError("Aucune donnee exploitable apres filtrage des classes.")\n\n    y_str = data[args.target_col].astype(str)\n    drop_cols = [c for c in [args.id_col, args.target_col] if c in data.columns]\n    X = data.drop(columns=drop_cols).copy()\n\n    if X.shape[1] == 0:\n        raise RuntimeError("Aucune feature disponible apres preparation des donnees.")\n\n    missing_before = float(X.isna().mean().mean())\n    X = interpolate_by_index(X)\n    missing_after = float(X.isna().mean().mean())\n\n    encoder = LabelEncoder()\n    y = encoder.fit_transform(y_str)\n\n    X_train, X_test, y_train, y_test = train_test_split(\n        X,\n        y,\n        test_size=args.test_size,\n        random_state=args.random_state,\n        stratify=y,\n    )\n\n    train_size_before_aug = len(y_train)\n    X_train, y_train = augment_train_data(\n        X_train=X_train,\n        y_train=y_train,\n        method=args.augment,\n        random_state=args.random_state,\n        k_neighbors=args.smote_k_neighbors,\n    )\n    train_size_after_aug = len(y_train)\n\n    class_counts_train = np.bincount(y_train)\n    n_classes = len(class_counts_train)\n    n_samples = len(y_train)\n    class_weights = n_samples / (n_classes * class_counts_train)\n    sample_weight = class_weights[y_train]\n\n    best_params = {}\n    cv_macro_mean = None\n    cv_macro_std = None\n    cv_source = "disabled"\n\n    if args.tune_n_iter > 0 and args.cv_folds >= 2:\n        cv = StratifiedKFold(n_splits=args.cv_folds, shuffle=True, random_state=args.random_state)\n        tune_model = make_xgb_model(\n            n_classes=n_classes,\n            random_state=args.random_state,\n            n_jobs=1,\n            max_depth=args.max_depth,\n            min_child_weight=args.min_child_weight,\n            gamma=args.gamma,\n            reg_alpha=args.reg_alpha,\n            reg_lambda=args.reg_lambda,\n        )\n        param_dist = {\n            "n_estimators": [300, 500, 700, 900],\n            "learning_rate": [0.02, 0.03, 0.05, 0.08],\n            "max_depth": [4, 6, 8, 10],\n            "min_child_weight": [1, 2, 4, 6],\n            "subsample": [0.6, 0.7, 0.8, 1.0],\n            "colsample_bytree": [0.6, 0.7, 0.8, 1.0],\n            "gamma": [0.0, 0.1, 0.2, 0.5],\n            "reg_alpha": [0.0, 0.1, 0.5, 1.0],\n            "reg_lambda": [0.5, 1.0, 2.0, 5.0],\n        }\n        search = RandomizedSearchCV(\n            estimator=tune_model,\n            param_distributions=param_dist,\n            n_iter=args.tune_n_iter,\n            scoring="f1_macro",\n            n_jobs=args.tune_n_jobs,\n            cv=cv,\n            random_state=args.random_state,\n            verbose=args.tune_verbose,\n            refit=True,\n            pre_dispatch=max(1, args.tune_n_jobs) if args.tune_n_jobs != -1 else "2*n_jobs",\n        )\n        try:\n            search.fit(X_train, y_train, sample_weight=sample_weight)\n        except OSError as exc:\n            if "WinError 1450" in str(exc) and args.tune_n_jobs != 1:\n                print("WinError 1450 detecte pendant le tuning, retry automatique avec --tune-n-jobs=1")\n                search.set_params(n_jobs=1, pre_dispatch=1)\n                search.fit(X_train, y_train, sample_weight=sample_weight)\n            else:\n                raise\n        best_params = search.best_params_\n        cv_macro_mean = float(search.best_score_)\n        cv_macro_std = float(search.cv_results_["std_test_score"][search.best_index_])\n        cv_source = "randomized_search_cv"\n        print(f"Tuning termine | Best CV Macro-F1: {cv_macro_mean:.4f} (+/- {cv_macro_std:.4f})")\n    elif args.cv_folds >= 2:\n        cv_source = "stratified_kfold_baseline"\n\n    model = make_xgb_model(\n        n_classes=n_classes,\n        random_state=args.random_state,\n        n_jobs=args.train_n_jobs,\n        max_depth=args.max_depth,\n        min_child_weight=args.min_child_weight,\n        gamma=args.gamma,\n        reg_alpha=args.reg_alpha,\n        reg_lambda=args.reg_lambda,\n        overrides=best_params,\n    )\n\n    if cv_source == "stratified_kfold_baseline":\n        cv_macro_mean, cv_macro_std = stratified_cv_macro_f1(\n            model=model,\n            X_train=X_train,\n            y_train=y_train,\n            sample_weight=sample_weight,\n            cv_folds=args.cv_folds,\n            random_state=args.random_state,\n        )\n        print(f"CV Macro-F1 (baseline): {cv_macro_mean:.4f} (+/- {cv_macro_std:.4f})")\n\n    X_fit, X_val, y_fit, y_val, w_fit, _ = train_test_split(\n        X_train,\n        y_train,\n        sample_weight,\n        test_size=args.val_size,\n        random_state=args.random_state,\n        stratify=y_train,\n    )\n\n    fit_kwargs = {\n        "sample_weight": w_fit,\n        "eval_set": [(X_val, y_val)],\n        "verbose": False,\n    }\n    early_stopping_mode = "disabled"\n    if args.early_stopping_rounds > 0:\n        try:\n            # Compatible avec les versions recentes: parametre sur l\'estimateur.\n            model.set_params(early_stopping_rounds=args.early_stopping_rounds)\n            early_stopping_mode = "estimator_param"\n        except ValueError:\n            try:\n                # Fallback pour certaines versions qui passent par callback.\n                import xgboost as xgb\n\n                fit_kwargs["callbacks"] = [\n                    xgb.callback.EarlyStopping(rounds=args.early_stopping_rounds, save_best=True)\n                ]\n                early_stopping_mode = "callback"\n            except Exception:\n                early_stopping_mode = "unsupported"\n\n    model.fit(X_fit, y_fit, **fit_kwargs)\n\n    y_pred = model.predict(X_test)\n\n    acc = accuracy_score(y_test, y_pred)\n    macro_accuracy = balanced_accuracy_score(y_test, y_pred)\n    macro_precision = precision_score(y_test, y_pred, average="macro", zero_division=0)\n    macro_recall = recall_score(y_test, y_pred, average="macro", zero_division=0)\n    macro_f1 = f1_score(y_test, y_pred, average="macro")\n    weighted_f1 = f1_score(y_test, y_pred, average="weighted")\n\n    class_names = encoder.classes_\n    report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True, zero_division=0)\n    report_df = pd.DataFrame(report).T\n\n    cm = confusion_matrix(y_test, y_pred)\n    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)\n    cm_row_sum = cm_df.sum(axis=1).replace(0, np.nan)\n    df_norm = cm_df.div(cm_row_sum, axis=0).fillna(0.0)\n\n    feature_names = X.columns.tolist()\n    feat_imp, date_imp, index_imp = aggregate_feature_importance(model, feature_names)\n\n    metrics = {\n        "n_samples": int(len(data)),\n        "n_features": int(X.shape[1]),\n        "n_classes": int(len(class_names)),\n        "augmentation": args.augment,\n        "phenology_enabled": bool(not args.disable_phenology),\n        "train_size_before_augmentation": int(train_size_before_aug),\n        "train_size_after_augmentation": int(train_size_after_aug),\n        "cv_folds": int(args.cv_folds),\n        "cv_source": cv_source,\n        "val_size": float(args.val_size),\n        "early_stopping_rounds": int(args.early_stopping_rounds),\n        "early_stopping_mode": early_stopping_mode,\n        "cv_macro_f1_mean": None if cv_macro_mean is None else float(cv_macro_mean),\n        "cv_macro_f1_std": None if cv_macro_std is None else float(cv_macro_std),\n        "best_params": best_params,\n        "missing_before_imputation": missing_before,\n        "missing_after_imputation": missing_after,\n        "accuracy": float(acc),\n        "macro_accuracy": float(macro_accuracy),\n        "macro_precision": float(macro_precision),\n        "macro_recall": float(macro_recall),\n        "macro_f1": float(macro_f1),\n        "weighted_f1": float(weighted_f1),\n    }\n\n    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")\n    report_df.to_csv(out_dir / "classification_report.csv", index=True)\n    cm_df.to_csv(out_dir / "confusion_matrix.csv", index=True)\n    df_norm.to_csv(out_dir / "confusion_matrix_normalized.csv", index=True)\n    feat_imp.to_csv(out_dir / "feature_importance_gain.csv", index=False)\n    date_imp.to_csv(out_dir / "date_importance_gain.csv", index=False)\n    index_imp.to_csv(out_dir / "index_importance_gain.csv", index=False)\n\n    plt.figure(figsize=(8, 6))\n    plt.imshow(df_norm, cmap="Greens")\n    plt.colorbar()\n    plt.xticks(range(len(class_names)), class_names, rotation=90)\n    plt.yticks(range(len(class_names)), class_names)\n    plt.xlabel("Classe prédite")\n    plt.ylabel("Classe réelle")\n    plt.title("Matrice de confusion (normalisée)")\n    ax = plt.gca()\n    ax.set_frame_on(False)\n    for spine in ax.spines.values():\n        spine.set_visible(False)\n    ax.tick_params(length=0)\n    plt.tight_layout()\n    plt.savefig(out_dir / "confusion_matrix_normalized.png", dpi=200)\n    plt.close()\n\n    print("\\n=== Resultats XGBoost ===")\n    print(f"Samples: {metrics[\'n_samples\']:,} | Features: {metrics[\'n_features\']:,} | Classes: {metrics[\'n_classes\']}")\n    print(f"Augmentation: {metrics[\'augmentation\']} | Train avant/apres: {metrics[\'train_size_before_augmentation\']:,}/{metrics[\'train_size_after_augmentation\']:,}")\n    print(f"Phenology: {metrics[\'phenology_enabled\']}")\n    if metrics["cv_macro_f1_mean"] is not None:\n        print(f"CV Macro-F1: {metrics[\'cv_macro_f1_mean\']:.4f} (+/- {metrics[\'cv_macro_f1_std\']:.4f}) [{metrics[\'cv_source\']}]")\n    print(f"Missing (avant): {metrics[\'missing_before_imputation\']:.4f}")\n    print(f"Missing (apres): {metrics[\'missing_after_imputation\']:.4f}")\n    print(f"Accuracy: {metrics[\'accuracy\']:.4f}")\n    print(f"Macro-Accuracy: {metrics[\'macro_accuracy\']:.4f}")\n    print(f"Macro-Precision: {metrics[\'macro_precision\']:.4f}")\n    print(f"Macro-Recall: {metrics[\'macro_recall\']:.4f}")\n    print(f"Macro-F1: {metrics[\'macro_f1\']:.4f}")\n    print(f"Weighted-F1: {metrics[\'weighted_f1\']:.4f}")\n    print(f"\\nFichiers ecrits dans: {out_dir.resolve()}")\n\n\nif __name__ == "__main__":\n    main()\n'

xgb_module = types.ModuleType('xgb_embedded')
xgb_module.__file__ = '<embedded:train_xgboost.py>'
sys.modules['xgb_embedded'] = xgb_module
exec(XGBOOST_SOURCE, xgb_module.__dict__)

print('Module embarque charge: xgb_embedded')


## Cellule 4 - Helpers d'exécution et lecture des résultats

Fonctions utilitaires pour lancer une variante et lire `metrics.json`.


In [ ]:
import json
from pathlib import Path


def run_embedded_main(module, argv: list[str]) -> None:
    old_argv = sys.argv[:]
    try:
        sys.argv = ["train_xgboost.py"] + [str(x) for x in argv]
        module.main()
    finally:
        sys.argv = old_argv


def run_xgboost_variant(argv: list[str]) -> None:
    run_embedded_main(xgb_module, argv)


def read_metrics(run_dir: Path) -> dict:
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Fichier introuvable: {metrics_path}")
    return json.loads(metrics_path.read_text(encoding="utf-8"))


def print_metrics(metrics: dict) -> None:
    keys = ["accuracy", "macro_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]
    for key in keys:
        if key in metrics:
            print(f"{key}: {metrics[key]:.4f}")


## Cellule 5 - Variantes

Toutes les variantes sont définies ici, avec la **variante finale recommandée** explicitement marquée.


In [ ]:
RANDOM_STATE = 42
TRAIN_N_JOBS = 4
TUNE_N_JOBS = 1

VARIANTES = {
    "v1_baseline": {
        "description": "Baseline simple (parametres par defaut).",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--output-dir", str(OUTPUT_ROOT / "baseline"),
        ],
    },
    "v2_rapide": {
        "description": "Rapide: sans tuning (tune-n-iter=0).",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--tune-n-iter", "0",
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "rapide"),
        ],
    },
    "v3_finale_recommandee": {
        "description": "VARIANTE FINALE RECOMMANDEE: tuning modere (n_iter=5, cv=3), stable sous Windows.",
        "is_final": True,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--tune-n-iter", "5",
            "--cv-folds", "3",
            "--tune-n-jobs", str(TUNE_N_JOBS),
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "final_recommandee"),
        ],
    },
    "v4_tuning_complet": {
        "description": "Tuning complet (tres lent).",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--tune-n-iter", "15",
            "--cv-folds", "5",
            "--tune-n-jobs", str(TUNE_N_JOBS),
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "tuning_complet"),
        ],
    },
    "v5_smote": {
        "description": "SMOTE + tuning modere.",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--augment", "smote",
            "--tune-n-iter", "5",
            "--cv-folds", "3",
            "--tune-n-jobs", str(TUNE_N_JOBS),
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "smote"),
        ],
    },
    "v6_borderline_smote": {
        "description": "Borderline-SMOTE + tuning modere.",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--augment", "borderline_smote",
            "--tune-n-iter", "5",
            "--cv-folds", "3",
            "--tune-n-jobs", str(TUNE_N_JOBS),
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "borderline_smote"),
        ],
    },
    "v7_sans_phenologie": {
        "description": "Sans features phenologiques.",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--disable-phenology",
            "--tune-n-iter", "5",
            "--cv-folds", "3",
            "--tune-n-jobs", str(TUNE_N_JOBS),
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "sans_phenology"),
        ],
    },
    "v8_rapide_max_dates_30": {
        "description": "Mode rapide avec reduction du nombre de dates (max-dates=30).",
        "is_final": False,
        "args": [
            "--indices-csv", str(INDICES_CSV),
            "--labels-csv", str(LABELS_CSV),
            "--max-dates", "30",
            "--tune-n-iter", "0",
            "--train-n-jobs", str(TRAIN_N_JOBS),
            "--output-dir", str(OUTPUT_ROOT / "max_dates_30"),
        ],
    },
}

VARIANTE_FINALE = "v3_finale_recommandee"

for nom, meta in VARIANTES.items():
    suffix = " <- VARIANTE FINALE" if meta.get("is_final", False) else ""
    print(f"{nom}: {meta['description']}{suffix}")


## Cellule 6 - Exécuter une variante

Choisis la variante à lancer. Les métriques macro sont affichées à la fin.


In [ ]:
VARIANTE_A_LANCER = VARIANTE_FINALE

if VARIANTE_A_LANCER not in VARIANTES:
    raise ValueError(f"Variante inconnue: {VARIANTE_A_LANCER}")

meta = VARIANTES[VARIANTE_A_LANCER]
args = list(meta["args"])

print("Lancement de:", VARIANTE_A_LANCER)
run_xgboost_variant(args)

run_dir = Path(meta["args"][meta["args"].index("--output-dir") + 1])
print("\nDossier run:", run_dir)
if run_dir.exists():
    metrics = read_metrics(run_dir)
    print("\n=== Metriques ===")
    print_metrics(metrics)


## Cellule 7 - Comparer plusieurs runs

Renseigne les dossiers de runs pour construire un tableau comparatif.


In [ ]:
import pandas as pd

RUNS_A_COMPARER = [
    OUTPUT_ROOT / "baseline",
    OUTPUT_ROOT / "rapide",
    OUTPUT_ROOT / "final_recommandee",
]

rows = []
for run in RUNS_A_COMPARER:
    metrics_path = Path(run) / "metrics.json"
    if not metrics_path.exists():
        continue
    m = json.loads(metrics_path.read_text(encoding="utf-8"))
    rows.append(
        {
            "run": str(run),
            "accuracy": m.get("accuracy"),
            "macro_accuracy": m.get("macro_accuracy"),
            "macro_precision": m.get("macro_precision"),
            "macro_recall": m.get("macro_recall"),
            "macro_f1": m.get("macro_f1"),
            "weighted_f1": m.get("weighted_f1"),
        }
    )

if rows:
    df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
    display(df)
else:
    print("Aucun metrics.json trouv? dans la liste RUNS_A_COMPARER.")
